# 02.05 Encoding and Numerical Preprocessing

**Phase 02 — Data Preparation**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Formulate scikit-learn compatible preprocessing transformers for categorical encoding, numerical scaling, and power transformations.


### Research Objectives

1. Define encoders for nominal features (One-Hot) and ordinal features (Ordinal/Target Encoding where justified).
2. Apply appropriate scaling (StandardScaler, RobustScaler) for skewed financial and numerical features.
3. Encapsulate preprocessing in leak-proof pipelines that can be fit strictly on training folds.
4. Serialize transformer specifications for deployment consistency.


### Inputs & Expected Outputs

**Required Inputs:**
- `Cleaned patient tabular records (`data/interim/cleaned/`).`

**Expected Outputs:**
- `Preprocessing pipeline specification (`models/pretrained/tabular_preprocessor_spec.json`).`
- `Preprocessed interim arrays for baseline analysis.`


### 1. Environment Setup & Library Imports

In [ ]:
from pathlib import Path
import json, datetime, re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

### 1. Setup Environment and Column Definitions

In [ ]:
CLEANED_CSV = PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'ofphil_cleaned_tabular.csv'
SPLITS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'splits'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'features'
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessing' / 'numerical_scaling_stats.csv'
for d in [SPLITS_DIR / 'train', SPLITS_DIR / 'validation', SPLITS_DIR / 'test', FEATURES_DIR, AUDIT_CSV_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CLEANED_CSV)

### 2. Define Feature Subsets (Numerical vs Categorical)

In [ ]:
num_features = ['household_size', 'number_of_dependents', 'monthly_family_income', 'monthly_household_obligations_expenses', 'cost_of_treatment', 'total_supporting_documents']
cat_features = ['employment_status', 'residence_status', 'educational_attainment', 'diagnosis_severity', 'type_of_treatment_surgery', 'assistance_type']
X = df[num_features + cat_features].copy()
y = (df['eligibility'] == 'ELIGIBLE').astype(int)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=20261123)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=20261123)
print(f'Splits -> train={len(X_train)}, val={len(X_val)}, test={len(X_test)}')

### 3. Build Leak-Proof ColumnTransformer

In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features),
])
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)
joblib.dump(preprocessor, FEATURES_DIR / 'preprocessing_pipeline.joblib')
X_train.to_csv(SPLITS_DIR / 'train' / 'X_train.csv', index=False); y_train.to_csv(SPLITS_DIR / 'train' / 'y_train.csv', index=False)
X_val.to_csv(SPLITS_DIR / 'validation' / 'X_val.csv', index=False); y_val.to_csv(SPLITS_DIR / 'validation' / 'y_val.csv', index=False)
X_test.to_csv(SPLITS_DIR / 'test' / 'X_test.csv', index=False); y_test.to_csv(SPLITS_DIR / 'test' / 'y_test.csv', index=False)

### 4. Validation and Serialization of Transformation Rules

In [ ]:
stats_rows = []
for col in num_features:
    s = pd.to_numeric(X_train[col], errors='coerce').dropna()
    stats_rows.append({
        'feature_name': col, 'split_scope': 'train',
        'mean': round(float(s.mean()), 4), 'std': round(float(s.std()), 4),
        'min': round(float(s.min()), 4), 'max': round(float(s.max()), 4),
        'scaler_type': 'RobustScaler',
    })
pd.DataFrame(stats_rows).to_csv(AUDIT_CSV_PATH, index=False)
print(f'Saved numerical scaling stats -> {AUDIT_CSV_PATH}')

### Findings & Observations

- **Observations:** Document scaling sensitivity, handling of unseen categories, and skewness of socioeconomic variables requiring robust scaling.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Categorical and numerical feature subsets strictly separated.
- [ ] RobustScaler selected for skewed socioeconomic indicators.
- [ ] OneHotEncoder configured with handle_unknown='ignore'.
- [ ] Pipeline formulated to prevent test fold contamination.
